<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/neuron_finder/neuron_finder.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Finding, Steering and Tuning Neurons: neuron-finder in Interpretune

neuron-finder <a id="cite-1"></a>[[1]](#ref-1) finds the MLP neurons that induce a behaviour: it screens every neuron,
steers the strongest candidates one at a time at a range of doses, and keeps those whose effect exceeds what random
neurons steered the same way produce. This notebook runs that workflow with Interpretune's surfaces, and then does what
the found neurons are for:

1. **Screen** the neurons of Qwen3-0.6B for one behaviour.
2. **Steer** each candidate against random controls, and keep the ones that beat them.
3. **Save** the found neurons as a checkpoint address book: which units, in which checkpoint, with the doses in a
   treatment beside them.
4. **Steer from the book** on two backends, nnsight and TransformerLens, and check that they agree.
5. **Import** neuron-finder's own `layer:neuron:dose` lists as books.
6. **Tune only the found neurons**, with unit adapters that a Fine-Tuning Scheduler schedule thaws by role, and compare
   tuning with steering and with tuning random neurons.

**The behaviour** is the model's answer disposition on RTE: whether it answers Yes (the premise entails the
hypothesis) or No. It is read as the answer margin, the logit of Yes minus the logit of No at the answer position,

$$
m = z_{\text{Yes}} - z_{\text{No}}
$$

so $m \gt 0$ is a Yes. This is the simple first step of [#869](https://github.com/speediedan/interpretune/issues/869):
it finds neurons that move the answer, not neurons that represent entailment. That question needs contrast pairs, and is
the next step.

Requirements: `uv pip install -e ".[examples]"` and one GPU with 16 GB.

> **Until the `speediedan/rte` component is public (#261), this notebook needs a Hugging Face token with access to
> it** (set `HF_TOKEN`). Qwen3-0.6B itself is ungated.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
N_SCREEN = 96  # validation items the screens read
N_EVAL = 96  # held-out validation items every effect is measured on
TOP = 4  # candidates per target (toward Yes, toward No)
DOSES = [4.0, 8.0, 16.0]  # in units of each neuron's own activation standard deviation
N_CONTROLS = 16  # random neurons steered toward random targets, per dose
BOOK_DOSE = 8.0  # the dose the book records
LIMIT_TRAIN_BATCHES = 40  # training batches per epoch, each stage
LIMIT_VAL_BATCHES = 8
ADAPTER_LR = 1.0e-3
SEED = 0
REVISION = (
    None  # component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [4]:
# @title Imports { display-mode: "form" }
import os
import random

import torch
import yaml
from nnsight import TransformersModel
from transformer_lens.model_bridge import TransformerBridge

import interpretune as it
from interpretune.adapters.nnsight.backends import NNsightModelBackend
from interpretune.adapters.transformer_lens.backends import TLModelBackend
from interpretune.config import SessionRunnerCfg
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.weights import attach_unit_adapter
from it_examples.seeds import ensure_local_seeds
from it_examples.utils import neuron_finder as nf
from it_examples.utils.nb_ui_utils import display_notebook_table

torch.manual_seed(SEED)
random.seed(SEED)

## 1. The Model and the Behaviour

The session is the `rte_ft.qwen3.core` configuration of the `speediedan/rte` component: Qwen3-0.6B in fp32 on one GPU,
answering RTE through Interpretune's core loop. Its chat template is rendered with thinking disabled, so the answer
position holds Yes or No rather than the start of a reasoning block. The validation split is cut into a screen set and a
held-out evaluation set; every effect below is measured on the evaluation set.

In [5]:
# @title 1: The session and the untouched model's answers { display-mode: "form" }
MODEL_ID = "Qwen/Qwen3-0.6B"
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
if REVISION is None:  # a validation run passes a local-publish REVISION and needs none of these steps
    it.hub.pull("speediedan/rte")  # caches the component's manifest and entrypoint; executes nothing
    os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running its entrypoint, for this session only
    it.hub.pull("speediedan/rte", "rte_ft.qwen3.core")


def rte_session(module_cls=None, lr=None):
    """A session on the component's configuration, optionally with a subclassed module and learning rate."""
    itdm_cfg, it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_ft.qwen3.core", revision=REVISION)
    if lr is not None:
        it_cfg.optimizer_init["init_args"]["lr"] = lr
    session = it.ITSession(
        it.ITSessionConfig(
            adapter_ctx=(it.Adapter.core,),
            datamodule_cfg=itdm_cfg,
            module_cfg=it_cfg,
            datamodule_cls=dm_cls,
            module_cls=module_cls(m_cls) if module_cls is not None else m_cls,
        )
    )
    it.it_init(**session)
    return session


session = rte_session()
module, tokenizer = session.module, session.datamodule.tokenizer
hf_model = module.model.eval()
REVISION_OF_MODEL = nf.cached_revision(MODEL_ID)  # the checkpoint the neuron indices below belong to
ANSWERS = tuple(int(i) for i in module.it_cfg.entailment_mapping_indices)  # (Yes, No)
N_LAYERS, WIDTH = hf_model.config.num_hidden_layers, hf_model.config.intermediate_size
batches = list(session.datamodule.val_dataloader())
per_batch = batches[0]["input_ids"].shape[0]
screen_batches = batches[: N_SCREEN // per_batch]
eval_batches = batches[N_SCREEN // per_batch : (N_SCREEN + N_EVAL) // per_batch]
eval_labels = torch.cat([b["labels"].view(-1) for b in eval_batches])


def answers(margins: torch.Tensor) -> dict:
    """Mean margin, Yes rate and accuracy (label 0 is entailment, answered Yes)."""
    predicted = (margins <= 0).long()
    return {
        "mean margin": f"{margins.mean():+.2f}",
        "answers Yes": f"{(margins > 0).float().mean():.0%}",
        "accuracy": f"{(predicted == eval_labels).float().mean():.1%}",
    }


@torch.no_grad()
def held_out_margins(model) -> torch.Tensor:
    """The answer margin of every held-out item, from a plain forward pass."""
    margins = []
    for b in eval_batches:
        logits = model(input_ids=b["input_ids"].cuda(), attention_mask=b["attention_mask"].cuda()).logits
        margins.append(nf.answer_margin(logits, ANSWERS).cpu())
    return torch.cat(margins)


base = answers(held_out_margins(hf_model))
print(f"{MODEL_ID} at {REVISION_OF_MODEL[:12]}: {N_LAYERS} layers of {WIDTH} MLP neurons")
display_notebook_table(
    [list(base.values())],
    list(base),
    title=f"The untouched model on {len(eval_labels)} held-out RTE items",
    caption=f"{(eval_labels == 0).float().mean():.0%} of these items are entailed.",
)

src/interpretune/hub/components.py:579: 'speediedan/rte' resolved to the local-publish snapshot localafa1da5, which shadows the cached Hub revision(s) ['14d388e1b6a2', '22152f0e349b', '4db2f48ce89f', '67fff0a10dd0', '738e41229e39', '7d0bbe4d887c', '848038b1310a', '8f61b54c4d2c', 'a9b2407a96e5', 'b10c7651eb32', 'b1a920fc1400']; the Hub was not consulted (resolution is cache-only). If you meant the published artifact, run interpretune.hub.pull('speediedan/rte') or resolve with require_hub=True.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Qwen3Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Qwen3Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `Qwen3ForCausalLM.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `Qwen3ForCausalLM.forward`, you can safely ignore this message.


... (6 outputs left out)

[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


Qwen/Qwen3-0.6B at c1899de289a0: 28 layers of 3072 MLP neurons


mean margin,answers Yes,accuracy
+3.53,98%,60.4%


## 2. Screening the Neurons

A neuron is an input of an MLP's down projection: Interpretune's site `point:mlp.out.hook_in`, whose units are the
3,072 hidden neurons of each of the 28 layers. The down projections are found by address through the module's view
(`component:mlp.out & layers:ℓ & leaf:weight`), so no module path is spelled here.

Two screens read every neuron at the answer position over the screen set:

- **The write screen** is neuron-finder's: the gradient of the margin with respect to the neuron's activation,
  averaged over the items, $g_{\ell,u} = \mathbb{E}[\partial m / \partial a_{\ell,u}]$. It says how far a small edit of
  the neuron moves the answer.
- **The read screen** asks whether the neuron's activation differs between entailed and other items:
  $(\bar a^{+} - \bar a^{-}) / \sigma_{\ell,u}$, where $\sigma_{\ell,u}$ is the neuron's standard deviation over the
  screen set. It is shown beside the write screen, not used to rank.

Doses are in units of $\sigma_{\ell,u}$, so a dose of 4 moves a neuron by four times its own typical variation.
Candidates are ranked by $g_{\ell,u} \sigma_{\ell,u}$, the first-order effect of a dose of one, and the strongest `TOP`
toward each answer are kept.

In [6]:
# @title 2: Both screens, and the candidates { display-mode: "form" }
projections = nf.down_projections(module, N_LAYERS)
screened = nf.screen(hf_model, projections, screen_batches, ANSWERS)
sigma = nf.activation_std(screened.activations)
read = screened.read()
candidates = nf.select_candidates(screened.write * sigma, TOP)
TARGETS = {1: "Yes", -1: "No"}
display_notebook_table(
    [
        [
            TARGETS[target],
            f"{layer}:{unit}",
            f"{screened.write[layer, unit]:+.3f}",
            f"{sigma[layer, unit]:.3f}",
            f"{screened.write[layer, unit] * sigma[layer, unit]:+.3f}",
            f"{read[layer, unit]:+.2f}",
        ]
        for target, members in candidates.items()
        for layer, unit in members
    ],
    ["Toward", "Neuron (layer:unit)", "Write screen g", "σ", "g σ", "Read screen z"],
    title=f"The candidates, from {len(screened.labels)} screened items",
    caption=f"The largest read-screen |z| over all {N_LAYERS * WIDTH:,} neurons is {read.abs().max():.2f}.",
)

Toward,Neuron (layer:unit),Write screen g,σ,g σ,Read screen z
Yes,27:2268,+0.099,3.847,+0.382,-0.45
Yes,27:840,+0.085,2.956,+0.252,-0.84
Yes,17:1650,+0.404,0.579,+0.234,+0.75
Yes,20:1960,+0.153,1.493,+0.228,+0.70
No,21:1341,-0.113,2.802,-0.317,-0.76
No,19:2595,-0.127,1.713,-0.218,-0.85
No,25:1665,-0.089,2.138,-0.190,-0.85
No,19:826,-0.132,1.287,-0.170,-1.03


## 3. Steering Each Candidate, Against Random Controls

Each candidate is steered alone: a dose $d$ adds $s d \sigma_{\ell,u}$ to the neuron at the answer position, where the
edit's sign $s$ is the target's ($+1$ toward Yes, $-1$ toward No) times the sign of $g_{\ell,u}$, so the edit moves the
margin toward the target to first order. Its **signed effect** is the mean change of the margin on the held-out items,
times the target.

**The controls** are `N_CONTROLS` random neurons, each steered the same way toward a random target. A candidate is
**found** at a dose when its signed effect exceeds the controls' 95th percentile at that dose.

Every setting is steered through the path the book below uses: a one-setting address book, read as a recipe, translated
into a backend payload (`recipe_interventions`) and run on nnsight's model backend, which wraps the session's model.

In [7]:
# @title 3: The dose sweep { display-mode: "form" }
lm = TransformersModel(hf_model, tokenizer=tokenizer)  # nnsight, around the same weights
nnsight_backend = NNsightModelBackend.for_model(lm)
sizes = {(layer, unit): float(sigma[layer, unit]) for layer in range(N_LAYERS) for unit in range(WIDTH)}


def steer(settings, backend=nnsight_backend, model=lm, input_key="input_ids", device=None):
    """Steer the held-out items with a book of ``settings``; returns the clean and steered margins."""
    book = nf.settings_book(
        settings, model_id=MODEL_ID, revision=REVISION_OF_MODEL, statistic="activation_std", tokens="last"
    )
    payload = nf.book_payload(book, WIDTH, sizes)
    return nf.steered_margins(backend, model, eval_batches, payload, ANSWERS, input_key=input_key, device=device)


def signed_effect(layer, unit, target, dose):
    setting = nf.Setting(layer, unit, nf.edit_sign(screened.write, layer, unit, target) * dose)
    clean, steered = steer([setting])
    return target * (steered - clean).mean().item(), int(((clean > 0) != (steered > 0)).sum())


rng = random.Random(SEED)
controls = [(rng.randrange(N_LAYERS), rng.randrange(WIDTH), rng.choice((1, -1))) for _ in range(N_CONTROLS)]
sweep = {}
for dose in DOSES:
    control_effects = [signed_effect(layer, unit, target, dose)[0] for layer, unit, target in controls]
    effects = {
        (layer, unit, target): signed_effect(layer, unit, target, dose)
        for target, members in candidates.items()
        for layer, unit in members
    }
    beat = nf.found({key: value[0] for key, value in effects.items()}, control_effects)
    sweep[dose] = {"effects": effects, "found": beat, "p95": torch.quantile(torch.tensor(control_effects), 0.95).item()}

display_notebook_table(
    [
        [TARGETS[target], f"{layer}:{unit}"]
        + [
            f"{sweep[d]['effects'][(layer, unit, target)][0]:+.2f} ({sweep[d]['effects'][(layer, unit, target)][1]})"
            + (" ✓" if sweep[d]["found"][(layer, unit, target)] else "")
            for d in DOSES
        ]
        for target, members in candidates.items()
        for layer, unit in members
    ]
    + [["controls", "95th percentile"] + [f"{sweep[d]['p95']:+.3f}" for d in DOSES]],
    ["Toward", "Neuron", *(f"dose {d:g}σ" for d in DOSES)],
    title="Signed effect on the margin (answers flipped), each candidate steered alone",
    caption="✓: found, above the controls' 95th percentile at that dose.",
)

Toward,Neuron,dose 4σ,dose 8σ,dose 16σ
Yes,27:2268,+1.52 (2) ✓,+3.04 (2) ✓,+6.01 (2) ✓
Yes,27:840,+1.01 (2) ✓,+2.03 (2) ✓,+4.07 (2) ✓
Yes,17:1650,+0.90 (2) ✓,+1.64 (2) ✓,+2.70 (2) ✓
Yes,20:1960,+0.90 (2) ✓,+1.78 (2) ✓,+3.41 (2) ✓
No,21:1341,+1.31 (3) ✓,+2.68 (26) ✓,+5.35 (83) ✓
No,19:2595,+0.96 (2) ✓,+2.01 (12) ✓,+4.16 (61) ✓
No,25:1665,+0.73 (2) ✓,+1.42 (3) ✓,+2.67 (23) ✓
No,19:826,+0.71 (2) ✓,+1.44 (5) ✓,+2.94 (35) ✓
controls,95th percentile,+0.048,+0.096,+0.193


## 4. The Found Neurons as a Checkpoint Address Book

The untouched model answers Yes to nearly everything, so the useful direction is toward No. The neurons found toward No
at `BOOK_DOSE` go into a **checkpoint address book**. A neuron index means something in one checkpoint only, so the
book's `applies_to` pins the model id and the revision the screens ran on. The book separates what from how much:

- **the set** `found` lists the units at `point:mlp.out.hook_in`, per layer;
- **the treatment** `steer` adds each neuron's signed dose at the last token, and its `statistic` transform says the
  dose is in units of `activation_std`. The statistic is computed from the model, so whoever applies the book
  supplies it; here it is the screen's $\sigma$.

The book is plain YAML, and venngram reads and checks it when it is loaded.

In [8]:
# @title 4: The book { display-mode: "form" }
found_toward = {
    target: [(layer, unit) for layer, unit in members if sweep[BOOK_DOSE]["found"][(layer, unit, target)]]
    for target, members in candidates.items()
}
book_settings = [
    nf.Setting(layer, unit, nf.edit_sign(screened.write, layer, unit, -1) * BOOK_DOSE)
    for layer, unit in found_toward[-1]
]
assert book_settings, "no neuron toward No beat the controls at BOOK_DOSE"
book = nf.settings_book(
    book_settings,
    model_id=MODEL_ID,
    revision=REVISION_OF_MODEL,
    architecture="Qwen3ForCausalLM",
    name="qwen3-0.6b-rte-toward-no",
    statistic="activation_std",
    tokens="last",
)
print(yaml.safe_dump(book, sort_keys=False))
assert nf.book_settings(book) == sorted(book_settings), "the book reads back as the settings it was written from"

schema_version: 1
name: qwen3-0.6b-rte-toward-no
description: 'Steering settings (layer:neuron:dose): 21:1341:8, 19:2595:8, 25:1665:8,
  19:826:8'
portability: checkpoint
applies_to:
  architectures:
  - Qwen3ForCausalLM
  models:
  - id: Qwen/Qwen3-0.6B
    revision: c1899de289a04d12100db370d81485cdf75e47ca
    basis: unprocessed
sets:
  found:
    point: mlp.out.hook_in
    units:
      19:
      - 826
      - 2595
      21:
      - 1341
      25:
      - 1665
treatments:
  steer:
    model:
      id: Qwen/Qwen3-0.6B
      revision: c1899de289a04d12100db370d81485cdf75e47ca
    stages:
    - site: mlp.out.hook_in
      tokens: last
      ops:
      - op: add
        target:
          set: found
        value:
          doses:
            19:
              826: 8.0
              2595: 8.0
            21:
              1341: 8.0
            25:
              1665: 8.0
          transform:
          - statistic:
              name: activation_std



## 5. Steering From the Book on Two Backends

The same book is applied on nnsight, which wraps the session's model, and on TransformerLens, whose bridge loads its own
copy of the checkpoint. Both receive one payload, keyed by the site's TransformerLens spelling
(`blocks.ℓ.mlp.out.hook_in`), which each backend maps to its own module. They must agree on the clean margins and on the
steered ones.

In [9]:
# @title 5: nnsight and TransformerLens, one book { display-mode: "form" }
clean_nn, steered_nn = steer(book_settings)
bridge = TransformerBridge.boot_transformers(MODEL_ID, device="cuda", dtype=torch.float32)
clean_tl, steered_tl = steer(book_settings, backend=TLModelBackend(), model=bridge, input_key="input", device="cuda")
del bridge
torch.cuda.empty_cache()
gaps = {"clean": (clean_nn - clean_tl).abs().max().item(), "steered": (steered_nn - steered_tl).abs().max().item()}
steered_from_book = answers(steered_nn)
display_notebook_table(
    [
        ["nnsight", *answers(clean_nn).values(), *answers(steered_nn).values()],
        ["TransformerLens", *answers(clean_tl).values(), *answers(steered_tl).values()],
    ],
    ["Backend", *(f"clean {k}" for k in base), *(f"steered {k}" for k in base)],
    title=f"The book's {len(book_settings)} neurons at {BOOK_DOSE:g}σ, on two backends",
    caption=f"Largest margin difference between the backends: clean {gaps['clean']:.4f}, "
    f"steered {gaps['steered']:.4f}.",
)
assert max(gaps.values()) < 0.05, f"the backends disagree: {gaps}"

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Backend,clean mean margin,clean answers Yes,clean accuracy,steered mean margin,steered answers Yes,steered accuracy
nnsight,+3.53,98%,60.4%,-3.76,0%,41.7%
TransformerLens,+3.53,98%,60.4%,-3.76,0%,41.7%


## 6. Importing neuron-finder's Lists

neuron-finder prints each setting it finds as `layer:neuron:dose`, with the dose in its own unit: a layer's median
residual norm over the norm of the neuron's down-projection column, $m_\ell / \lVert W_u \rVert_2$. A list of them
becomes a book the same way, with the statistic named `dose_unit`. `nf.residual_medians` measures $m_\ell$ on the screen
set, and `nf.dose_unit` turns it into each neuron's unit, so the imported list steers in activation units as
neuron-finder's did, at every token as neuron-finder steers. `nf.book_settings` reads a book back as the list, which is
the export direction.

The list below names two of the neurons found toward No, at doses chosen for this illustration rather than taken from a
neuron-finder run.

In [10]:
# @title 6: A neuron-finder list, imported and steered { display-mode: "form" }
imported = nf.settings_book(
    ["21:1341:0.05", "19:2595:0.05"], model_id=MODEL_ID, revision=REVISION_OF_MODEL, name="imported"
)
print("read back:", ", ".join(str(s) for s in nf.book_settings(imported)))


@torch.no_grad()
def screen_hidden_states(model):
    """Per screen batch, the model's hidden states (the embeddings, then one per layer)."""
    for b in screen_batches:
        out = model(
            input_ids=b["input_ids"].cuda(), attention_mask=b["attention_mask"].cuda(), output_hidden_states=True
        )
        yield tuple(h.cpu() for h in out.hidden_states)


medians = nf.residual_medians(screen_hidden_states(hf_model), [b["attention_mask"] for b in screen_batches])
members = [(s.layer, s.unit) for s in nf.book_settings(imported)]
# a Qwen3 down projection is an nn.Linear, so its units are columns (unit axis 1)
units = nf.dose_unit({layer: (projections[layer].weight, 1) for layer, _ in members}, medians, members)
clean, steered = nf.steered_margins(nnsight_backend, lm, eval_batches, nf.book_payload(imported, WIDTH, units), ANSWERS)
display_notebook_table(
    [["untouched", *answers(clean).values()], ["the imported list", *answers(steered).values()]],
    ["", *base],
    title="An imported neuron-finder list, steered at every token on nnsight",
    caption="Each neuron's dose unit, in activation units: "
    + ", ".join(f"{layer}:{unit} = {v:.3f}" for (layer, unit), v in sorted(units.items())),
)

read back: 19:2595:0.05, 21:1341:0.05


,mean margin,answers Yes,accuracy
untouched,+3.53,98%,60.4%
the imported list,+0.83,70%,74.0%


## 7. Tuning Only the Found Neurons

Steering adds a fixed dose. Tuning learns the change instead, from labelled data, and touches only the found neurons.
A **unit adapter** (`attach_unit_adapter`) is a trainable delta over the slices of the tensors a neuron is: neuron $u$
at `point:mlp.out.hook_in` is column $u$ of the down projection and row $u$ of the up and gate projections. The weights
become $W = W_0 + \Delta$, with $\Delta$ zero at first and nonzero only on the selected units, so the model is unchanged
until it trains, and the optimizer and checkpoints see an ordinary tensor.

A new session's module attaches two adapters before its view is built: `to_no` over the neurons found toward No, and
`to_yes` over those found toward Yes. Their tensors take the roles `adapter:to_no` and `adapter:to_yes`, and a
Fine-Tuning Scheduler schedule thaws them by role, in two experiment stages:

```python
{0: {"params": ["role:adapter:to_no"], "max_transition_epoch": 1}, 1: {"params": ["role:adapter:to_yes"]}}
```

Stage 0 tunes the neurons that move the answer toward No, the model's missing answer. Stage 1 adds the others. Every
other tensor stays frozen. The objective is RTE's own labels on the training split.

**The control** is the same run on random neurons: as many per adapter, in the same layers.

In [11]:
# @title 7: Two tuning runs, the found neurons and random ones { display-mode: "form" }
del lm, nnsight_backend, projections, module, hf_model, session
torch.cuda.empty_cache()


def by_layer(members):
    out = {}
    for layer, unit in members:
        out.setdefault(layer, []).append(unit)
    return out


def with_unit_adapters(adapters):
    """A module class that attaches one unit adapter per name, over its units in each layer."""

    def subclass(base):
        class UnitAdapted(base):
            def register_tunable_modules(self) -> None:
                super().register_tunable_modules()
                for name, per_layer in adapters.items():
                    for layer, units in sorted(per_layer.items()):
                        attach_unit_adapter(self, f"point:mlp.out.hook_in & layers:{layer}", units, name=name)

        return UnitAdapted

    return subclass


def tune(adapters):
    """Tune ``adapters`` with the two-stage schedule; returns the held-out margins after tuning."""
    torch.manual_seed(SEED)
    run = rte_session(with_unit_adapters(adapters), lr=ADAPTER_LR)
    roles = {role for role in run.module.tunable_view.roles().values() if role != "base"}
    assert roles == {"adapter:to_no", "adapter:to_yes"}, roles
    schedule = {
        0: {"params": ["role:adapter:to_no"], "max_transition_epoch": 1},
        1: {"params": ["role:adapter:to_yes"]},
    }
    fts = FTSCoreCallback(schedule)
    SessionRunner(
        run_cfg=SessionRunnerCfg(
            it_session=run,
            max_epochs=2,
            limit_train_batches=LIMIT_TRAIN_BATCHES,
            limit_val_batches=LIMIT_VAL_BATCHES,
            callbacks=[fts],
        )
    ).train()
    assert fts.tuning_scheduler.curr_depth == 1, "both stages must run"
    model = run.module.model.eval()
    trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
    margins = held_out_margins(model)
    del run, model
    torch.cuda.empty_cache()
    return margins, trained


found_adapters = {"to_no": by_layer(found_toward[-1]), "to_yes": by_layer(found_toward[1])}
rng = random.Random(SEED + 1)
random_adapters = {
    name: by_layer([(layer, rng.randrange(WIDTH)) for layer, units in per_layer.items() for _ in units])
    for name, per_layer in found_adapters.items()
}
tuned_found, n_found = tune(found_adapters)
tuned_random, n_random = tune(random_adapters)

src/interpretune/hub/components.py:579: 'speediedan/rte' resolved to the local-publish snapshot localafa1da5, which shadows the cached Hub revision(s) ['14d388e1b6a2', '22152f0e349b', '4db2f48ce89f', '67fff0a10dd0', '738e41229e39', '7d0bbe4d887c', '848038b1310a', '8f61b54c4d2c', 'a9b2407a96e5', 'b10c7651eb32', 'b1a920fc1400']; the Hub was not consulted (resolution is cache-only). If you meant the published artifact, run interpretune.hub.pull('speediedan/rte') or resolve with require_hub=True.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Qwen3Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Qwen3Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `Qwen3ForCausalLM.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `Qwen3ForCausalLM.forward`, you can safely ignore this message.


... (47 outputs left out)


Epoch 0 end: {'train_loss': 0.5887, 'val_loss': 0.7085, 'accuracy': 0.6250}


[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule



Epoch 1 end: {'train_loss': 0.6056, 'val_loss': 0.6419, 'accuracy': 0.6094}


In [12]:
# @title Steering and tuning, side by side { display-mode: "form" }
display_notebook_table(
    [
        ["untouched", "", *base.values()],
        [f"steered from the book ({BOOK_DOSE:g}σ)", "", *steered_from_book.values()],
        ["tuned: the found neurons", f"{n_found:,}", *answers(tuned_found).values()],
        ["tuned: random neurons, same layers", f"{n_random:,}", *answers(tuned_random).values()],
    ],
    ["", "Trained parameters", *base],
    title=f"{len(eval_labels)} held-out RTE items",
    caption=f"Adapters: {found_adapters} (found), {random_adapters} (random).",
)

,Trained parameters,mean margin,answers Yes,accuracy
untouched,,+3.53,98%,60.4%
steered from the book (8σ),,-3.76,0%,41.7%
tuned: the found neurons,"24,576",+0.45,66%,76.0%
"tuned: random neurons, same layers","24,576",-0.06,43%,74.0%


## What This Showed

On 96 held-out RTE items, Qwen3-0.6B with this prompt answers Yes almost always, so its accuracy is the share of entailed
items. The rendered numbers are in the tables above; the pattern they show:

- **The screen finds neurons that move the answer, and the controls say how much that means.** Every candidate's signed
  effect is well above what random neurons steered the same way produce, at every dose. The strongest neurons toward No
  flip most answers at the highest dose.
- **The book carries the result across backends.** Units, checkpoint and doses are one document; nnsight and
  TransformerLens apply it to the same margins.
- **Steering sets the answer threshold.** The book's $8 \sigma$ dose at the last token overshoots: the model answers No to
  nearly everything, which is no better at the task. The imported list's much smaller dose at every token moves the
  model only part of the way, and its accuracy rises well above the untouched model's. The untouched model ranks the
  items better than its answers show, and much of the gain, steered or tuned, comes from correcting its bias toward
  Yes.
- **Tuning the found neurons** learns from the labels and lands near the best steered accuracy. Tuning as many random
  neurons in the same layers does about as well, so for learning the task, the screen's choice of neurons matters less
  than having a few trainable units in those layers. The found neurons are what move the answer; they are not shown to
  be the best neurons to tune.

**Next** ([#869](https://github.com/speediedan/interpretune/issues/869)): neurons that respond to entailment itself,
found by contrast on minimal pairs, an item and the same item with its evidence removed, and persona-style contrast
through `prompt_analysis`.

## References

1. <a id="ref-1"></a>changjonathanc. *neuron-finder*.
   [github.com/changjonathanc/neuron-finder](https://github.com/changjonathanc/neuron-finder) [↩](#cite-1)